# 04. Customer Clustering

Apply K-Means clustering to identify distinct customer personas.

**Output:** `data/processed/customer_segments.csv`

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
from src.clustering import run_clustering_analysis, find_optimal_k, prepare_features

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## Load RFM Data

In [ ]:
rfm_df = pd.read_csv('../data/processed/rfm_scores.csv')
print(f"Loaded RFM data for {len(rfm_df):,} customers")
rfm_df.head()

## Feature Preparation

In [ ]:
# Features for clustering
feature_cols = ['recency', 'frequency', 'monetary', 'avg_order_value']
scaled_features, scaler, features = prepare_features(rfm_df, feature_cols)

print(f"\nFeature Statistics (before scaling):")
rfm_df[feature_cols].describe().round(2)

## Finding Optimal K

In [ ]:
results = find_optimal_k(
    scaled_features,
    k_range=range(2, 11),
    save_plot='../reports/figures/cluster_evaluation.png'
)

## Apply K-Means Clustering

In [ ]:
# Choose optimal K based on silhouette score
optimal_k = results['k'][np.argmax(results['silhouette'])]
print(f"Selected K = {optimal_k} based on silhouette analysis")

# Run full clustering pipeline
clustered_df, profiles = run_clustering_analysis(
    rfm_df,
    n_clusters=optimal_k,
    save_path='../data/processed/customer_segments.csv',
    plots_dir='../reports/figures'
)

In [ ]:
print("\n" + "=" * 60)
print("CLUSTER PROFILES")
print("=" * 60)
profiles

## Cluster Visualization (PCA)

In [ ]:
# Reduce to 2D for visualization
pca = PCA(n_components=2)
pca_features = pca.fit_transform(scaled_features)

print(f"Explained variance ratio: {pca.explained_variance_ratio_.sum():.2%}")

fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter(
    pca_features[:, 0], 
    pca_features[:, 1], 
    c=clustered_df['cluster'],
    cmap='viridis',
    alpha=0.6,
    s=30
)

# Add cluster centroids
for cluster in clustered_df['cluster'].unique():
    mask = clustered_df['cluster'] == cluster
    centroid = pca_features[mask].mean(axis=0)
    label = clustered_df[mask]['cluster_label'].iloc[0]
    ax.scatter(centroid[0], centroid[1], marker='X', s=200, c='red', edgecolor='black', linewidth=2)
    ax.annotate(label, (centroid[0], centroid[1]), fontsize=10, fontweight='bold',
                xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('Principal Component 1', fontsize=12)
ax.set_ylabel('Principal Component 2', fontsize=12)
ax.set_title('Customer Clusters (PCA Projection)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Cluster')

plt.tight_layout()
plt.savefig('../reports/figures/cluster_pca.png', dpi=150)
plt.show()

## Silhouette Analysis

In [ ]:
# Silhouette plot
silhouette_vals = silhouette_samples(scaled_features, clustered_df['cluster'])
clustered_df['silhouette'] = silhouette_vals

fig, ax = plt.subplots(figsize=(10, 8))

y_lower = 10
colors = plt.cm.viridis(np.linspace(0, 1, optimal_k))

for i in range(optimal_k):
    cluster_silhouette = silhouette_vals[clustered_df['cluster'] == i]
    cluster_silhouette.sort()
    y_upper = y_lower + len(cluster_silhouette)
    
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_silhouette,
                      facecolor=colors[i], alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * len(cluster_silhouette), str(i))
    y_lower = y_upper + 10

ax.axvline(x=silhouette_score(scaled_features, clustered_df['cluster']), color='red', linestyle='--',
           label=f"Avg: {silhouette_score(scaled_features, clustered_df['cluster']):.3f}")
ax.set_xlabel('Silhouette Coefficient')
ax.set_ylabel('Cluster')
ax.set_title('Silhouette Plot for Clusters', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/silhouette_plot.png', dpi=150)
plt.show()

## Cluster Comparison (Box Plots)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.flatten(), feature_cols):
    sns.boxplot(data=clustered_df, x='cluster_label', y=col, ax=ax, palette='husl')
    ax.set_xlabel('')
    ax.set_ylabel(col.replace('_', ' ').title())
    ax.set_title(f'{col.replace("_", " ").title()} by Cluster', fontweight='bold')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../reports/figures/cluster_boxplots.png', dpi=150)
plt.show()

## Cluster Radar Chart

In [ ]:
from math import pi

# Normalize metrics for radar
cluster_means = clustered_df.groupby('cluster_label')[feature_cols].mean()

# Normalize to 0-1 scale for comparison
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())

# Invert recency (lower is better)
cluster_means_norm['recency'] = 1 - cluster_means_norm['recency']

categories = ['Recency\n(inverted)', 'Frequency', 'Monetary', 'Avg Order\nValue']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
colors = sns.color_palette('husl', len(cluster_means_norm))

for i, (cluster, row) in enumerate(cluster_means_norm.iterrows()):
    values = row.values.tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=cluster, color=colors[i])
    ax.fill(angles, values, alpha=0.15, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('Cluster Profiles Comparison (Normalized)', fontsize=14, fontweight='bold', y=1.1)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1))

plt.tight_layout()
plt.savefig('../reports/figures/cluster_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## Final Customer Segments Summary

In [ ]:
print("=" * 70)
print("CUSTOMER SEGMENTS SUMMARY")
print("=" * 70)

for cluster_label in clustered_df['cluster_label'].unique():
    subset = clustered_df[clustered_df['cluster_label'] == cluster_label]
    print(f"\n🎯 {cluster_label}")
    print(f"   Customers: {len(subset):,} ({len(subset)/len(clustered_df)*100:.1f}%)")
    print(f"   Total Revenue: ${subset['monetary'].sum():,.0f}")
    print(f"   Avg Recency: {subset['recency'].mean():.0f} days")
    print(f"   Avg Frequency: {subset['frequency'].mean():.1f} orders")
    print(f"   Avg Monetary: ${subset['monetary'].mean():,.0f}")
    print(f"   Avg Order Value: ${subset['avg_order_value'].mean():.2f}")

In [ ]:
# Export summary for Power BI
cluster_summary = clustered_df.groupby('cluster_label').agg({
    'customer_id': 'count',
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': ['mean', 'sum'],
    'avg_order_value': 'mean'
}).round(2)
cluster_summary.columns = ['customer_count', 'avg_recency', 'avg_frequency', 'avg_monetary', 'total_revenue', 'avg_order_value']
cluster_summary['revenue_pct'] = (cluster_summary['total_revenue'] / cluster_summary['total_revenue'].sum() * 100).round(1)
cluster_summary.to_csv('../data/processed/cluster_summary.csv')

print("\n✅ Clustering Complete!")
print(f"📁 Customer segments: data/processed/customer_segments.csv")
print(f"📁 Cluster summary: data/processed/cluster_summary.csv")

In [ ]:
clustered_df.head(10)